In [ ]:
import os
import sys
import talib as ta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
%matplotlib inline
sns.set_theme()

In [ ]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from src.Alpha9.utility import get_config, read_file

In [ ]:
# read data for a specific crypto
i = 0
config = get_config.load()
symbols = config['pipeline']['symbols']
timeperiod_cat = config['pipeline']['timeperiod_cat']
timeperiod_cat0 = config['pipeline']['timeperiod_cat0']
timeperiod_cat1 = config['pipeline']['timeperiod_cat1']
timeperiod_cat2 = config['pipeline']['timeperiod_cat2']
timeperiod_cat3 = config['pipeline']['timeperiod_cat3']
timeperiod_cat4 = config['pipeline']['timeperiod_cat4']
data = read_file.read_data('raw', symbols[i])

In [ ]:
open_px = data['open']
high_px = data['high']
low_px = data['low']
close_px = data['close']
vol = data['volume']

In [ ]:
def ICHIMOKU(high_px, low_px, close_px, fastperiod, medperiod, slowperiod):
    tenkan = (ta.MAX(high_px, timeperiod=fastperiod) + ta.MIN(low_px, timeperiod=fastperiod)) / 2
    kijun = (ta.MAX(high_px, timeperiod=medperiod) + ta.MIN(low_px, timeperiod=medperiod)) / 2
    span_b = (ta.MAX(high_px, timeperiod=slowperiod) + ta.MIN(low_px, timeperiod=slowperiod)) / 2
    cloud_trend = (close_px - span_b) / close_px
    tk_cross = (tenkan - kijun) / close_px
    return cloud_trend, tk_cross

In [ ]:
def VWAP(high_px, low_px, close_px, vol, period):
    typical_price = (high_px + low_px + close_px) / 3
    pv = typical_price * vol
    pv_sum = pv.rolling(window=period, min_periods=period).sum()
    vol_sum = vol.rolling(window=period, min_periods=period).sum()

    return pv_sum / vol_sum

In [ ]:
# trend indicators
data['sma-dist'] = (close_px - ta.SMA(close_px, timeperiod=timeperiod_cat4)) / ta.SMA(close_px, timeperiod=timeperiod_cat4)
data['ema-dist'] = (close_px - ta.EMA(close_px, timeperiod=timeperiod_cat3)) / ta.EMA(close_px, timeperiod=timeperiod_cat3)
data['macd'], data['macd-signal'], data['macd-hist'] = ta.MACD(close_px, fastperiod=timeperiod_cat1, slowperiod=timeperiod_cat3, signalperiod=timeperiod_cat0)
data['trix'] = ta.TRIX(close_px, timeperiod=timeperiod_cat2)
data['sar'] = (close_px - ta.SAR(high_px, low_px)) / close_px
data['tema'] = (close_px - ta.TEMA(close_px, timeperiod=timeperiod_cat2)) / ta.TEMA(close_px, timeperiod=timeperiod_cat2)
data['trima'] = ta.TRIMA(close_px, timeperiod=timeperiod_cat4)
data['wma'] = ta.WMA(close_px, timeperiod=timeperiod_cat3)
data['dema'] = (close_px - ta.DEMA(close_px, timeperiod=timeperiod_cat4)) / ta.DEMA(close_px, timeperiod=timeperiod_cat4)
data['ppo'] = ta.PPO(close_px, fastperiod=timeperiod_cat1, slowperiod=timeperiod_cat3)
data['plus-di'] = ta.PLUS_DI(high_px, low_px, close_px, timeperiod=timeperiod_cat0)
data['minus-di'] = ta.MINUS_DI(high_px, low_px, close_px, timeperiod=timeperiod_cat0)
data['lin-reg-slope'] = ta.LINEARREG_SLOPE(close_px, timeperiod=timeperiod_cat2)
data['ichi-cloud-trend'], data['ichi-tk-cross'] = ICHIMOKU(high_px, low_px, close_px, timeperiod_cat0, timeperiod_cat3, timeperiod_cat4)

# momentum indicators
data['rsi'] = ta.RSI(close_px, timeperiod=timeperiod_cat2)
data['stoch-osc-slowk'], data['stoch-osc-slowd'] = ta.STOCH(high_px, low_px, close_px, fastk_period=timeperiod_cat2, slowk_period=timeperiod_cat, slowd_period=timeperiod_cat)
data['adx'] = ta.ADX(high_px, low_px, close_px, timeperiod=timeperiod_cat2)
data['momentum'] = ta.MOM(close_px, timeperiod=timeperiod_cat2)
data['cci'] = ta.CCI(high_px, low_px, close_px, timeperiod=timeperiod_cat2)
data['cmo'] = ta.CMO(close_px, timeperiod=timeperiod_cat2)
fastk, fastd = ta.STOCHRSI(close_px, timeperiod=timeperiod_cat2, fastk_period=timeperiod_cat2, fastd_period=timeperiod_cat)
data['stoch-rsi'] = fastk - fastd
data['williams-%r'] = ta.WILLR(high_px, low_px, close_px, timeperiod=timeperiod_cat2)
data['bop'] = ta.BOP(open_px, high_px, low_px, close_px)

# volatility indicators
data['natr'] = ta.NATR(high_px, low_px, close_px, timeperiod=timeperiod_cat2)
upper, middle, lower = ta.BBANDS(close_px, timeperiod=timeperiod_cat3)
data['bbands_pct'] = (close_px - lower) / (upper - lower)
data['bbands_width'] = (upper - lower) / middle
data['std-dev'] = ta.STDDEV(np.log(close_px), timeperiod=timeperiod_cat3)

# volume indicators
data['obv'] = ta.OBV(close_px, vol)
data['vwap'] = VWAP(high_px, low_px, close_px, vol, timeperiod_cat3)
data['mfi'] = ta.MFI(high_px, low_px, close_px, vol, timeperiod=timeperiod_cat2)
data['ad-line'] = ta.AD(high_px, low_px, close_px, vol)

In [ ]:
data.dropna(inplace=True)
data

In [ ]:
def plot_heatmap(corr_matrix):
    plt.figure(figsize=(20, 16))
    sns.heatmap(
        corr_matrix,
        annot=True,
        fmt=".2f",
        cmap='coolwarm',
        vmin=-1, vmax=1,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8}
    )

    plt.title("Correlation Matrix - Features", fontsize=16, color='blue')
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
def evaluate_feature(data, features):
    results = []

    # plot correlation heatmap
    plot_heatmap(data[features].corr())

    # calculate metrics
    for feature in features:
        result = {'Feature' : feature}

        momentum_indicators = ['rsi', 'stoch-osc', 'adx', 'momentum', 'cmo', 'stoch-rsi', 'williams-%r', 'bop']
        volatility_indicators = ['natr', 'bbands_pct', 'bbands_width', 'std-dev']
        volume_indicators = ['obv', 'vwap', 'mfi', 'ad-line']

        if feature in momentum_indicators:
            result['Type'] = 'Momentum'
        elif feature in volatility_indicators:
            result['Type'] = 'Volatility'
        elif feature in volume_indicators:
            result['Type'] = 'Volume'
        else:
            result['Type'] = 'Trend'

        # calculate redundancy score
        corr = data[features].corrwith(data[feature])
        others = corr.drop(feature)
        result['Redundancy Score'] = np.linalg.norm(others)

        # calculate stationarity
        adf = adfuller(data[feature].values)
        result['Stationarity Value'] = f'{adf[1]:.5}'

        results.append(result)

    return results

In [ ]:
features = list(data.columns)
removed = ['open', 'high', 'low', 'close', 'volume', 'minus-di']
features = [feature for feature in features if feature not in removed]
result = evaluate_feature(data, features)

In [ ]:
result_df = pd.DataFrame(result)
print(result_df)
print(result_df.describe())